In [ ]:
import numpy as np
import sisl
import matplotlib.pyplot as plt
from ase.visualize import view
from sisl import Hamiltonian
from tqdm.auto import tqdm

In [ ]:
def loop_GF(energies, eta, )

In [ ]:
bond = 1.43
gr = sisl.geom.graphene(bond=bond)
Ham0 = Hamiltonian(gr)
r = (0.1*bond, bond+1e-2)
t = (0.0, -2.7)
Ham0.construct([r, t])
Ham = Ham0.tile(1,0).tile(1,1)

Na = Nb = 12
eta = 1e-2j
nk1 = int(np.ceil(3*900/Nb))

rse = sisl.RealSpaceSE(Ham, 0, 1, (Na, Nb, 1))
rse.setup(eta=eta, bz=sisl.MonkhorstPack(Ham, [1, nk1, 1]))
HamNN = Ham.tile(Na, 0).tile(Nb, 1)
geomNN = HamNN.geometry
Ham_elec, elec_indices = rse.real_space_coupling(ret_indices=True)
nC = len(elec_indices)
HamNN.set_nsc([1,1,1])

all_atoms = np.arange(0, HamNN.na)
inside_atoms = np.delete(all_atoms, elec_indices, axis=None)
alist = np.concatenate([elec_indices, inside_atoms])

HamNN_reordered = HamNN.sub(alist)
HamNN_reordered.reduce()
H_sub = HamNN_reordered.Hk(format="array")
S_sub = HamNN_reordered.Sk(format="array")

In [ ]:
dE = 0.1
Emax = 3.0 # eV
Emin = -Emax
energies = np.arange(Emin, Emax+dE, dE)

nE = len(energies)
N = len(HamNN_reordered)

LDOS = np.empty(shape=(nE, N), dtype=float)


for idx, E in enumerate(tqdm(energies, desc="LDOS")):
    z = E + eta
    invG = z*S_sub - H_sub
    RSE = rse.self_energy(z)
    RSE_reordered = RSE[np.ix_(alist, alist)]
    invG[0:len(alist), 0:len(alist)] -= RSE_reordered
    G = np.linalg.inv(invG)
    diagG = np.diag(G)
    LDOS[idx, :] = -(1.0/np.pi) * np.imag(diagG)
    
LDOS_dev = LDOS[nC:, :]
DOS_dev = LDOS_dev.sum(axis=1)

In [ ]:
DATA = np.load("alans_calcs.npz")
DOS_OG = DATA["dos"]

In [ ]:
# === plot DOS ===
E_idx=31
E_idx=min(E_idx, len(energies)) -1

fig, ax = plt.subplots(1, 2, figsize=(6,4))

ax[0].plot(energies, DOS_OG, c="k", label="OG method")
ax[0].plot(energies, DOS_dev, c="r", label="new method")
ax[0].vlines(x=energies[E_idx],ymin=0,ymax=600,color='k',linestyle='dashed')


for a in ax:
    a.set_xlabel("Energy (eV)")
    a.set_ylabel("DOS (states / eV)")
    a.set(ylim=(0,100), xlim=(-3,3))
    a.legend()

ax[1].plot(energies, np.abs(DOS_OG - DOS_dev), label="Diff")
fig.tight_layout()
None